In [9]:
import wandb
from datetime import datetime, timezone

wandb.login()
api = wandb.Api()

# 1. Define your project path
project_path = "tidiane/patch_icl_eval"

# 2. Get today's starting timestamp in UTC (W&B uses UTC)
timestamp_str  = "2026-02-23T00:00:00Z"  # For testing purposes, you can hardcode a date
method = "universeg"

# 3. Fetch runs using the MongoDB-style filter
filtered_runs = api.runs(
    path=project_path,
    filters={"created_at": {"$gt": timestamp_str}, "config.method": method}
)

runs_data = {}

# 4. Iterate through the fetched runs and extract artifacts + metrics
for run in filtered_runs:
    print(f"Processing run: {run.name} (ID: {run.id})")
    
    try:
        # --- NEW: Extract your logged metrics ---

        
        # Optional: You can also just grab the entire summary dictionary 
        # if you want ALL logged metrics at once!
        # all_metrics = dict(run.summary) 

        # Dynamically construct the artifact path using the run.id
        artifact_path = f"{project_path}/run-{run.id}-per_case_dice:v0"
        artifact = api.artifact(artifact_path)
        artifact.download()

        # Extract the table
        table = artifact.get("per_case_dice.table.json")
        df = table.get_dataframe()
        
        # Store in your dictionary
        runs_data[run.name] = {
            "wandb_id": run.id,
            "config": run.config,
            "gflops_per_sample": run.summary.get("gflops_per_sample"), 
            "val_final_dice": run.summary.get("val_final_dice"),  # Example of another metric
            "per_case_dice": df
        }
    except wandb.errors.CommError:
        print(f"  -> No matching artifact found for run {run.name}")
    except Exception as e:
        print(f"  -> Error processing run {run.name}: {e}")

print(f"\nSuccessfully downloaded data for {len(runs_data)} runs.")

Processing run: silver-dawn-103 (ID: azy5wotf)


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


Processing run: confused-mountain-104 (ID: effsv0dn)


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


Processing run: efficient-pine-107 (ID: uaqa50ae)


...

Processing run: volcanic-shadow-111 (ID: 5mqkd5pw)


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


Processing run: true-fog-114 (ID: ocn1skod)


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  



Successfully downloaded data for 7 runs.


In [10]:
for run_name, data in runs_data.items():
    #print(f"\nRun: {run_name}")
    print(f"W&B ID: {data['wandb_id']}")
    print(f"context_size: {data['config']['context_size']}")
    print(f"input_size: {data['config']['model']['universeg']['input_size']}")
    print(f"gflops_per_sample: {data['gflops_per_sample']}")
    print(f"val_final_dice: {data['val_final_dice']}")
    #print("Per-case Dice DataFrame:")
    #print(data["per_case_dice"].head())

W&B ID: azy5wotf
context_size: 1
input_size: 128
gflops_per_sample: 13.572767744
val_final_dice: 0.07844752718624198
W&B ID: effsv0dn
context_size: 3
input_size: 128
gflops_per_sample: 34.334572544
val_final_dice: 0.18635937626592616
W&B ID: uaqa50ae
context_size: 15
input_size: 128
gflops_per_sample: 158.905401344
val_final_dice: 0.3481535982127452
W&B ID: lefvdage
context_size: 10
input_size: 64
gflops_per_sample: 26.750222336
val_final_dice: 0.25033283267104117
W&B ID: ek7xolx4
context_size: 10
input_size: 128
gflops_per_sample: 107.000889344
val_final_dice: 0.31396038318453745
W&B ID: 5mqkd5pw
context_size: 10
input_size: 256
gflops_per_sample: 428.003557376
val_final_dice: 0.2908167294533067
W&B ID: ocn1skod
context_size: 10
input_size: 512
gflops_per_sample: 1712.014229504
val_final_dice: 0.22743874287015903


In [11]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set publication-ready style
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.autolayout': True,
    'axes.linewidth': 1.5,
})

# Extracted data
data = [
    {'context_size': 1, 'input_size': 128, 'gflops': 13.57, 'dice': 0.0784},
    {'context_size': 3, 'input_size': 128, 'gflops': 34.33, 'dice': 0.1864},
    {'context_size': 10, 'input_size': 128, 'gflops': 107.00, 'dice': 0.3140}, 
    {'context_size': 15, 'input_size': 128, 'gflops': 158.91, 'dice': 0.3482},
    {'context_size': 10, 'input_size': 64, 'gflops': 26.75, 'dice': 0.2503},
    {'context_size': 10, 'input_size': 256, 'gflops': 428.00, 'dice': 0.2908},
    {'context_size': 10, 'input_size': 512, 'gflops': 1712.01, 'dice': 0.2274},
]

df = pd.DataFrame(data)

# Separate data for the two plots
df_context = df[df['input_size'] == 128].sort_values('context_size').copy()
df_input = df[df['context_size'] == 10].sort_values('input_size').copy()

def plot_dual_bar(df, x_col, x_label, title, filename):
    fig, ax1 = plt.subplots(figsize=(7, 5))
    
    # We need string labels for the x-axis to space bars evenly
    x_labels = df[x_col].astype(str).tolist()
    x = np.arange(len(x_labels))
    width = 0.35  # width of the bars

    # 1. Plot Dice on primary y-axis
    color1 = 'tab:blue'
    ax1.set_xlabel(x_label, fontweight='bold')
    ax1.set_ylabel('Validation Dice', color=color1, fontweight='bold')
    # Offset Dice bars to the left
    bar1 = ax1.bar(x - width/2, df['dice'], width, color=color1, label='Dice Score', alpha=0.85)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.set_ylim(0, 0.4) # Normalize range for easy visual comparison
    
    # Set x-ticks and labels
    ax1.set_xticks(x)
    ax1.set_xticklabels(x_labels)

    # 2. Instantiate a secondary axes that shares the same x-axis
    ax2 = ax1.twinx()
    
    # 3. Plot GFLOPs on secondary y-axis
    color2 = 'tab:red'
    ax2.set_ylabel('GFLOPs / Sample', color=color2, fontweight='bold')
    # Offset GFLOPs bars to the right
    bar2 = ax2.bar(x + width/2, df['gflops'], width, color=color2, label='GFLOPs', alpha=0.85)
    ax2.tick_params(axis='y', labelcolor=color2)

    # 4. Combine legends from both axes
    bars = [bar1, bar2]
    labels = [b.get_label() for b in bars]
    ax1.legend(bars, labels, loc='upper left')

    plt.title(title, pad=15, fontweight='bold')
    
    # Save the figure
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

# Create plots
plot_dual_bar(df_context, 'context_size', 'Context Size', 'Impact of Context Size\n(Fixed Input Size: 128)', 'context_size_bar.pdf')
plot_dual_bar(df_input, 'input_size', 'Input Size', 'Impact of Input Size\n(Fixed Context Size: 10)', 'input_size_bar.pdf')

In [ ]:
df

,case_id,label_id,axis,dice
0,s0000,colon,z,7.181328e-11
1,s0000,colon,z,1.060221e-10
2,s0000,colon,z,8.240626e-11
3,s0000,colon,z,8.597713e-11
4,s0000,colon,z,8.328475e-11
...,...,...,...,...
44552,s1428,vertebrae_T9,x,3.935069e-01
44553,s1428,vertebrae_T9,x,3.141492e-01
44554,s1428,vertebrae_T9,x,3.900252e-01
44555,s1428,vertebrae_T9,x,4.497476e-01
